# FraudTrace AI - 09: Comprehensive Model Evaluation & Golden Test Suite

> **DISCLAIMER: SYNTHETIC DATA ONLY**
> All data used in this notebook is synthetically generated for research, testing, and demonstration purposes. No real police records, personal identification, or banking transactions are included. This system does not make legal guilt determinations; it reconstructs and correlates evidentiary relationships.

### Purpose
Evaluate the trained contradiction classifier and verify all 8 official golden test cases.

| Attribute | Description |
| :--- | :--- |
| **Input** | Trained model artifact and ml/data/sample/golden_tests.json |
| **Output** | Evaluation metrics report and Golden Test Scorecard (8/8 pass) |
| **Target Environment** | Google Colab (CPU / GPU compatible) & Local |
| **Package** | `ml.src` |


In [ ]:
# ====================================================================
# GOOGLE COLAB ENVIRONMENT SETUP
# ====================================================================
import os
import sys

# Clone repository if running inside Google Colab
if 'google.colab' in str(get_ipython()):
    if not os.path.exists('FraudTrace.AI'):
        !git clone https://github.com/Cyberdude441/FraudTrace.AI.git
    %cd FraudTrace.AI
    !pip install -r ml/requirements-colab.txt

# Ensure repository root is on sys.path for local and Colab execution
repo_root = os.path.abspath('.')
if repo_root not in sys.path:
    sys.path.insert(0, repo_root)

print(f"[OK] Environment initialized. Working directory: {os.getcwd()}")


## 1. Load Trained Model Artifact
Load the serialized model from `ml/artifacts/models/contradiction_model_v1.pkl`.

In [ ]:
from ml.src.models.contradiction_model import ContradictionClassifier
from ml.src.config import get_artifacts_dir

model_path = get_artifacts_dir() / "models" / "contradiction_model_v1.pkl"
model = ContradictionClassifier.load(str(model_path))
print(f"[OK] Successfully loaded trained model from {model_path}")


## 2. Execute Official 8 Golden Tests Suite
Verify each test case rigorously against expected forensic outcomes.

In [ ]:
from ml.src.data.loader import DataLoader
from ml.src.inference.pipeline import ForensicInferencePipeline

loader = DataLoader(dataset_type="sample")
golden_tests = loader.load_golden_tests()
pipeline = ForensicInferencePipeline()

print(f"=== Executing {len(golden_tests)} Golden Tests ===\n")
results = []

for test in golden_tests:
    tid = test["test_id"]
    name = test["name"]
    inputs = test["input_evidence"]
    expected = test["expected_output"]
    
    output = pipeline.analyze_event(inputs)
    status_match = output["contradiction"]["status"] == expected["contradiction_status"]
    conf_match = output["confidence"]["level"] == expected["confidence_level"]
    passed = status_match and conf_match
    
    results.append({
        "ID": tid,
        "Name": name,
        "Status": output["contradiction"]["status"],
        "Expected Status": expected["contradiction_status"],
        "Confidence": output["confidence"]["level"],
        "Expected Conf": expected["confidence_level"],
        "Result": "PASS" if passed else "FAIL"
    })

import pandas as pd
df_results = pd.DataFrame(results)
print(df_results.to_string(index=False))

total_passed = sum(1 for r in results if r["Result"] == "PASS")
print(f"\nSummary: {total_passed}/{len(golden_tests)} Golden Tests Passed ({total_passed/len(golden_tests)*100:.0f}%)")
assert total_passed == len(golden_tests), "Not all golden tests passed!"
